In [2]:
import pandas as pd

In [3]:
import os
import yaml
from pathlib import Path
from typing import Optional

notebook_dir = Path(os.getcwd())
project_dir = notebook_dir.parent
data_dir = notebook_dir / "data/"

In [28]:
sc_cxg_ds = pd.read_csv(data_dir / "sc_cxg_ds.tsv", sep = "\t")

with (data_dir /  "sc_cxg_params.yaml").open("r") as f:
    sc_cxg_params = yaml.safe_load(f)

sc_cxg_ds.head(2)

,Unnamed: 0,soma_joinid,citation,collection_id,collection_name,collection_doi,collection_doi_label,dataset_id,dataset_version_id,dataset_title,dataset_h5ad_path,dataset_total_cell_count,doi,abstract,species,tissue,condition,disease
0,9,9,Publication: https://doi.org/10.1126/sciimmuno...,3a2af25b-2338-4266-aad3-aa8d07473f50,Single-cell analysis of human B cell maturatio...,10.1126/sciimmunol.abe6291,King et al. (2021) Sci. Immunol.,00ff600e-6e2e-4d76-846f-0eec4f0ae417,3fa969b4-25e8-4e0b-b2ec-86cd9ddd2ccb,Human tonsil nonlymphoid cells scRNA,00ff600e-6e2e-4d76-846f-0eec4f0ae417.h5ad,363,https://doi.org/10.1126/sciimmunol.abe6291,Integrated single-cell transcriptomic and BCR ...,human,blood,healthy,healthy
1,23,23,Publication: https://doi.org/10.1038/s41467-02...,bf325905-5e8e-42e3-933d-9a9053e9af80,Single-cell Atlas of common variable immunodef...,10.1038/s41467-022-29450-x,Rodríguez-Ubreva et al. (2022) Nat Commun,a5d95a42-0137-496f-8a60-101e17f263c8,f223d2b0-d878-443b-8bef-f182fe1f22d1,Steady-state B cells - scRNA-seq,a5d95a42-0137-496f-8a60-101e17f263c8.h5ad,1324,https://doi.org/10.1038/s41467-022-29450-x,Abstract Common variable immunodeficiency (CVI...,human,blood,immunodeficiency,immunodeficiency


In [38]:
def query_cellxgene_single_cell(
    *,
    species: str="",
    tissue: str="",
    disease: str=""
) -> str:
    vals = [("species", species), ("tissue", tissue), ("disease", disease)]
    for param, val in vals:
        val = val.lower()
        valid_vals = sc_cxg_params[param]
        if val and val not in valid_vals:
            return f"Error: invalid species name {val}; must be one of {valid_vals}"

    mask = pd.Series(True, index=sc_cxg_ds.index)
    for param, val in vals:
        if not val:
            continue
        mask &= sc_cxg_ds[param].eq(val)
    num = sum(mask)
    
    if num > 20:
        return f"Error: too many datasets match the search parameters"

    ret = "\n--\n".join(sc_cxg_ds[mask].apply(
        lambda row: f"citation: {row['citation']}\ndataset_id: {row['dataset_id']}\n{row['collection_name']}\n{row['dataset_title']}\n{row['abstract']}",
        axis = 1,
    ))
        
    return (f"Success: Here is a list of datasets matching {{species={species}, tissue={tissue}, disease={disease}}}: {ret}"
             "\n\nYou should manually determine which dataset is most relevant and identify the corresponding  dataset_id.")

In [35]:
print(query_cellxgene_single_cell(species="mouse", tissue="brain", disease="healthy"))

Success: Here is a list of datasets matching {species=mouse, tissue=brain, disease=healthy}: citation: Publication: https://doi.org/10.1038/s41593-022-01183-6 Dataset Version: https://datasets.cellxgene.cziscience.com/dcfe419f-237c-4c48-8e86-34e7f141e819.h5ad curated and distributed by CZ CELLxGENE Discover in Collection: https://cellxgene.cziscience.com/collections/0faa1af6-e504-4b88-a47b-69347e1bace5
dataset_id: 6347cc90-f284-41d8-a131-db4a37bd796f
Single-cell transcriptomics characterization of oligodendrocytes and microglia in white matter aging
Single-cell of aged oligodendrocytes
Abstract A hallmark of nervous system aging is a decline of white matter volume and function, but the underlying mechanisms leading to white matter pathology are unknown. In the present study, we found age-related alterations of oligodendrocyte cell state with a reduction in total oligodendrocyte density in aging murine white matter. Using single-cell RNA-sequencing, we identified interferon (IFN)-respon

In [37]:
print(query_cellxgene_single_cell(species="mouse", tissue="fuck", disease="healthy"))

Error: invalid species name fuck; must be one of ['brain', 'lung', 'heart', 'liver', 'kidney', 'pancreas', 'intestine', 'spleen', 'lymph_node', 'breast', 'uterus', 'placenta', 'blood', 'skin', 'muscle', 'adipose', 'eye', 'testis', 'ovary', 'thyroid', 'oral', 'unknown']
